# Base model evaluation
In this notebook, the performance of selected base models will be evaluated on long texts.

**Selected models:**
- Encoder-only:
    - XLM-RoBERTa-large (FacebookAI/xlm-roberta-large) (0.6B parameters)
- Decoder-based:
    - Qwen3-Embedding-0.6B (Qwen/Qwen3-Embedding-0.6B) (0.6B parameters)

In [1]:
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets.dataset_dict import DatasetDict
from datasets.arrow_dataset import Dataset
from datasets import load_dataset

import datasets
import os

## Baseline models evaluation

In [2]:
from transformers import AutoTokenizer, AutoModel

In [3]:
xml_roberta_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")

def chunk_text(text, chunk_size=512, overlap=62):
    token_ids = xml_roberta_tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    chunks = []
    start = 0
    while start < len(token_ids):
        end = start + chunk_size
        chunk_tokens = token_ids[start:end]
        chunk_text = xml_roberta_tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start += chunk_size - overlap

    return chunks

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
class XMLRoBERTa:

    name = "xlm-roberta-large"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("FacebookAI/xlm-roberta-large")
        self.tokenizer = AutoTokenizer.from_pretrained(
            "FacebookAI/xlm-roberta-large",
            dtype=torch.float16,
            device_map="auto")
        self.model.eval()

    def encode(
            self,
            inputs,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type = None,
            **kwargs) -> torch.tensor:

        embeddings = []

        # for each text in input make a chunking
        texts_chunked = [chunk_text(text, self.chunk_size) for text in inputs]
        texts_chunked_tokenized = []

        for text in texts_chunked:    # for each text in chunked input texts
            # perform a tokeniztaion of each chunk
            chunk_input_ids = []
            chunk_attention_masks = []

            tokenized = self.tokenizer(
                text,    # text is effectively a batch of chunks at this point
                padding=True,
                truncation=True,
                max_length=self.chunk_size,
                return_tensors="pt"
            )

            # get embeddings for all chunks
            with torch.no_grad():
                outputs = self.model(**tokenized)
                # mean pooling for each chunk
                embedding = outputs.last_hidden_state.mean(dim=1)
                # mean pooling across chunks:
                embedding = embedding.mean(dim=0)
                embeddings.append(embedding.unsqueeze(0).detach())

        return_embed = torch.cat(embeddings, dim=0)
        return return_embed


class Qwen3_Embedding:

    name = "Qwen3-Embedding-0.6B"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("Qwen/Qwen3-Embedding-0.6B",
                                               padding_side='left')
        self.tokenizer = AutoTokenizer.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            dtype=torch.float16,
            device_map="auto")

        self.model.eval()

    def __get_eos_token_embedding(self, last_hidden_states, attention_mask):
        return last_hidden_states[:, -1]

    def encode(
            self,
            inputs,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type = None,
            **kwargs) -> torch.tensor:

        embeddings = []

        # for each text in input make a chunking
        texts_chunked = [chunk_text(text, self.chunk_size) for text in inputs]
        texts_chunked_tokenized = []

        for text in texts_chunked:    # for each text in chunked input texts
            # perform a tokeniztaion of each chunk
            chunk_input_ids = []
            chunk_attention_masks = []

            tokenized = self.tokenizer(
                text,    # text is effectively a batch of chunks at this point
                padding=True,
                truncation=True,
                max_length=self.chunk_size,
                return_tensors="pt"
            )

            # get embeddings for all chunks
            with torch.no_grad():
                outputs = self.model(**tokenized)
                # mean pooling for each chunk
                embedding = outputs.pooler_output.mean(dim=1)
                embeddings.append(embedding.detach())

        return_embed = torch.cat(embeddings, dim=0)
        return return_embed

### MTEB LongEmbed benchmark

In [1]:
!pip install mteb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 13.7 MB/s eta 0:00:00


In [6]:
import mteb

long_embed = mteb.get_benchmark("LongEmbed")